##  1. Configuration et Imports

In [1]:
import pandas as pd
import numpy as np
import os
import re
from typing import Dict, Tuple, Optional, List
import warnings
warnings.filterwarnings('ignore')

# Configuration des chemins
BASE_PATH = '/home/henintsoa/CFIM'
CSV_PATH = os.path.join(BASE_PATH, 'csv')
FINAL_PATH = os.path.join(BASE_PATH, 'final/data')

print(" Imports chargés")
print(f"\n Chemins configurés:")
print(f"   • Base: {BASE_PATH}")
print(f"   • CSV: {CSV_PATH}")
print(f"   • Final: {FINAL_PATH}")

 Imports chargés

 Chemins configurés:
   • Base: /home/henintsoa/CFIM
   • CSV: /home/henintsoa/CFIM/csv
   • Final: /home/henintsoa/CFIM/final/data


##  2. Chargement des Données Météo

Chargeons les données météo extraites dans la Phase 1.1.

In [2]:
# Essayer de charger le fichier consolidé, sinon utiliser l'existant
fichier_consolide = os.path.join(FINAL_PATH, 'meteo_marine_cotiere_complet.csv')
fichier_existant = os.path.join(CSV_PATH, 'marine_cotiere_2019_2020.csv')

if os.path.exists(fichier_consolide):
    df_meteo = pd.read_csv(fichier_consolide)
    source = fichier_consolide
else:
    df_meteo = pd.read_csv(fichier_existant)
    source = fichier_existant

print(" DONNÉES MÉTÉO CHARGÉES")
print("=" * 70)
print(f"Source: {source}")
print(f"Nombre d'enregistrements: {len(df_meteo)}")
print(f"Colonnes: {list(df_meteo.columns)}")

print("\n Aperçu des données brutes:")
display(df_meteo.head(10))

 DONNÉES MÉTÉO CHARGÉES
Source: /home/henintsoa/CFIM/final/data/meteo_marine_cotiere_complet.csv
Nombre d'enregistrements: 1346
Colonnes: ['date', 'zone', 'vent', 'etat_mer', 'temps']

 Aperçu des données brutes:


,date,zone,vent,etat_mer,temps
0,06/09/2019,CAP D'AMBRE A MAHANORO,10/15 kt atteignant 20/25 kt au nord d'Antalaha,agitée à forte,Pluies faible a modérée
1,06/09/2019,MAHANORO AU CAP SAINTE MARIE,05/10 kt devenant progressivement secteur sud ...,agitée à forte,pluies
2,06/09/2019,CAP D'AMBRE A BESALAMPY,"15/20 kt localement 20 kt au sud de Majunga, v...",Non spécifié,Temps sec
3,06/09/2019,BESALAMPY A MOROMBE,15/20 kt,"agitée a forte, très forte près Morombe dans l...",Temps sec
4,06/09/2019,MOROMBE A CAP SAINTE MARIE,20/25 kt atteignant 30/35 kt entre Morombe et,Non spécifié,Temps partiellement nuageux
5,07/09/2019,CAP D'AMBRE A BESALAMPY,DE SUD-EST 15/20 KT ATTEIGNANT 30 KT LE MATIN ...,PEU AGITÉ,Non specifie
6,07/09/2019,BESALAMPY A MOROMBE,DE SECTEUR SUD 20/25 KT. LOCALEMENT 30 KT AU C...,AGITÉ,SEC
7,07/09/2019,MOROMBE AU CAP SAINTE MARIE,DE SUD-EST 20/25 KT TOURNANT SECTEUR EST TEMPO...,F,PARTIELLEMENT NUAGEUX
8,07/09/2019,MAHANORO AU CAP SAINTE MARIE,DE SUD-EST 20/25 KT ATTEIGNANT LOCALEMENT 30KT,"AGITÉE À FORTE, TRES FORTE PAR",Non specifie
9,07/09/2019,CAP D'AMBRE A MAHANORO,DE SUD-EST 15/20 KT ATTEIGNANT 25/30 KT AU NOR...,AGITÉE À FORTE. HAUTEUR DE VAGUE 2.8/3.2M,Non specifie


##  3. Analyse des Formats de Données

Avant de standardiser, analysons les **différents formats** rencontrés dans chaque variable.

In [3]:
==================
# ANALYSE DES FORMATS DE VENT
==================

print(" ANALYSE DES FORMATS DE VENT")
print("=" * 70)

# Échantillon des valeurs de vent
vent_samples = df_meteo['vent'].dropna().unique()[:20]
print("\n Exemples de valeurs:")
for i, v in enumerate(vent_samples, 1):
    print(f"   {i:2d}. {v[:80]}..." if len(str(v)) > 80 else f"   {i:2d}. {v}")

print("\n Patterns identifiés:")
print("""
   • "15/20 kt" → Plage de vitesse (min/max)
   • "atteignant 25 kt" → Rafales
   • "localement 30 kt" → Pointes locales
   • "secteur sud" → Direction
   • "variable" → Direction variable
""")

 ANALYSE DES FORMATS DE VENT

 Exemples de valeurs:
    1. 10/15 kt atteignant 20/25 kt au nord d'Antalaha
    2. 05/10 kt devenant progressivement secteur sud 20/25kt localement 30kt vers midi
    3. 15/20 kt localement 20 kt au sud de Majunga, variable 05/10kt ailleurs
    4. 15/20 kt
    5. 20/25 kt atteignant 30/35 kt entre Morombe et
    6. DE SUD-EST 15/20 KT ATTEIGNANT 30 KT LE MATIN ENTRE BESALAMPY ET ANALALAVA. VENT...
    7. DE SECTEUR SUD 20/25 KT. LOCALEMENT 30 KT AU COURS DE L'APRES-MIDI, S'AFFAIBLISS...
    8. DE SUD-EST 20/25 KT TOURNANT SECTEUR EST TEMPORAIRE 27/33 KT ENTRE MOROMBE ET AM...
    9. DE SUD-EST 20/25 KT ATTEIGNANT LOCALEMENT 30KT
   10. DE SUD-EST 15/20 KT ATTEIGNANT 25/30 KT AU NORD D'ANTALAHA
   11. DE SUD-EST 05/10 KT LOCALEMENT 15 KT AU SUD D'ANTALAHA; AILLEURS, SUD-EST 15/20 ...
   12. DE D'EST À NORD-EST 05/10 KT LOCALEMENT 15/20 KT DEVENANT NORD-EST 20/25 KT LOCA...
   13. DE SUD-EST 10/15 KT LOCALEMENT 20 KT TOURNANT SUD-OUEST 10/15 KT L'APRES- MID

In [4]:

print(" ANALYSE DES FORMATS D'ÉTAT DE LA MER")
print("=" * 70)

# Valeurs uniques d'état de la mer
etats_mer = df_meteo['etat_mer'].value_counts()
print("\n Valeurs les plus fréquentes:")
print(etats_mer.head(20))

print("\n Échelle de Douglas (état de la mer):")
print("""
   1. Calme (glassy)     - Hauteur 0 m
   2. Calme (rippled)    - Hauteur 0-0.1 m
   3. Belle              - Hauteur 0.1-0.5 m
   4. Peu agitée         - Hauteur 0.5-1.25 m
   5. Agitée             - Hauteur 1.25-2.5 m
   6. Forte              - Hauteur 2.5-4 m
   7. Très forte         - Hauteur 4-6 m
   8. Grosse             - Hauteur 6-9 m
   9. Énorme             - Hauteur > 14 m
""")

 ANALYSE DES FORMATS D'ÉTAT DE LA MER

 Valeurs les plus fréquentes:
etat_mer
peu agitée à agitée                       142
Non spécifié                              123
agitée à forte                            103
A                                          51
belle à peu agitée                         48
PEU                                        41
peu agitée                                 38
agitée                                     36
AGITÉ                                      18
belle                                      13
AGITÉE À FORTE PAR                         11
PEU AGITÉ                                  10
FORTE SOUS ORAGES                           9
belle à peu agitée localement agitée        8
belle à peu agitée, agitée sous grains      7
peu agitée, agitée sous grains              7
F                                           7
AGITÉE, 2/2.5M                              5
forte à très forte                          5
FORTE, 3/4M                                 5
Na

In [5]:

print(" ANALYSE DES FORMATS DE TEMPS")
print("=" * 70)

# Valeurs uniques de temps
temps_values = df_meteo['temps'].value_counts()
print("\n Valeurs les plus fréquentes:")
print(temps_values.head(20))

 ANALYSE DES FORMATS DE TEMPS

 Valeurs les plus fréquentes:
temps
Non spécifié                                     324
Non specifie                                      82
Temps sec                                         56
Temps peu nuageux                                 53
SENSIBLE : PARTIELLEMENT NUAGEUX. CAP             40
Partiellement nuageux.                            29
Pluies faibles                                    27
Temps partiellement nuageux                       25
PEU NUAGEUX                                       25
Peu nuageux.                                      22
Ensoleillé.                                       21
SENSIBLE : PLUIES FAIBLES. CAP                    19
Averses orageuses.                                18
nuageux ailleurs                                  16
SENSIBLE : PLUIES FAIBLES INTERMITTENTES. CAP     16
Averses isolees.                                  16
Pluies                                            15
SENSIBLE : PLUIES INTERMITTENTES

##  4. Standardisation du Vent

### Règles de conversion:
- Extraire les valeurs numériques des vitesses en **nœuds (kt)**
- Identifier les **rafales** ("atteignant", "localement")
- Extraire la **direction** si présente

In [6]:

class VentParser:
    """
    Parse les descriptions textuelles de vent et extrait
    les valeurs numériques standardisées.
    """
    
    # Mapping des directions
    DIRECTIONS = {
        'N': 0, 'NORD': 0,
        'NE': 45, 'NORD-EST': 45, 'NORD EST': 45,
        'E': 90, 'EST': 90,
        'SE': 135, 'SUD-EST': 135, 'SUD EST': 135,
        'S': 180, 'SUD': 180,
        'SW': 225, 'SO': 225, 'SUD-OUEST': 225, 'SUD OUEST': 225,
        'W': 270, 'O': 270, 'OUEST': 270,
        'NW': 315, 'NO': 315, 'NORD-OUEST': 315, 'NORD OUEST': 315
    }
    
    def __init__(self):
        # Pattern pour les vitesses: "15/20 kt" ou "15 kt"
        self.pattern_vitesse = r'(\d{1,2})(?:/(\d{1,2}))?\s*(?:kt|KT|noeud|NOEUD)'
        
        # Pattern pour les rafales: "atteignant 25 kt" ou "localement 30 kt"
        self.pattern_rafale = r'(?:atteignant|localement|pouvant atteindre|jusqu\'?[àa])\s*(\d{1,2})(?:/(\d{1,2}))?\s*(?:kt|KT)?'
        
        # Pattern pour la direction
        self.pattern_direction = r'(?:secteur|direction|de|du)?\s*(nord|sud|est|ouest|N|S|E|W|NE|SE|SW|NW|NO|SO)[\-\s]*(est|ouest)?'
    
    def parser(self, texte: str) -> Dict:
        """
        Parse une description de vent.
        
        Args:
            texte: Description textuelle du vent
            
        Returns:
            Dict avec vitesse_min, vitesse_max, rafales, direction
        """
        resultat = {
            'vent_vitesse_min': None,
            'vent_vitesse_max': None,
            'vent_rafales': None,
            'vent_direction': None,
            'vent_direction_deg': None
        }
        
        if not texte or pd.isna(texte) or texte == 'Non spécifié':
            return resultat
        
        texte = str(texte).upper()
        
        # Extraire les vitesses
        vitesses = re.findall(self.pattern_vitesse, texte, re.IGNORECASE)
        if vitesses:
            # Prendre la première correspondance
            v_min, v_max = vitesses[0]
            resultat['vent_vitesse_min'] = int(v_min)
            resultat['vent_vitesse_max'] = int(v_max) if v_max else int(v_min)
        
        # Extraire les rafales
        rafales = re.findall(self.pattern_rafale, texte, re.IGNORECASE)
        if rafales:
            r_min, r_max = rafales[0]
            resultat['vent_rafales'] = int(r_max) if r_max else int(r_min)
        
        # Extraire la direction
        directions = re.findall(self.pattern_direction, texte, re.IGNORECASE)
        if directions:
            dir_principale, dir_secondaire = directions[0]
            direction = dir_principale.upper()
            if dir_secondaire:
                direction = direction + '-' + dir_secondaire.upper()
            
            resultat['vent_direction'] = direction
            
            # Convertir en degrés
            for key, deg in self.DIRECTIONS.items():
                if direction.replace('-', ' ').replace(' ', '') == key.replace('-', ' ').replace(' ', ''):
                    resultat['vent_direction_deg'] = deg
                    break
        
        return resultat

print(" Classe VentParser chargée")

 Classe VentParser chargée


In [7]:
parser_vent = VentParser()

# Tests
tests_vent = [
    "10/15 kt atteignant 20/25 kt au nord d'Antalaha",
    "05/10 kt localement 15 kt",
    "15/20 kt localement 20 kt au sud de Majunga, variable 05/10kt ailleurs",
    "secteur sud 20/25kt localement 30kt",
    "vent de Sud-est 15/20 kt",
    "Non spécifié"
]

print(" TEST DU PARSER DE VENT")
print("=" * 70)

for test in tests_vent:
    result = parser_vent.parser(test)
    print(f"\n Entrée: \"{test[:60]}...\"" if len(test) > 60 else f"\n Entrée: \"{test}\"")
    print(f"   → Min: {result['vent_vitesse_min']} kt")
    print(f"   → Max: {result['vent_vitesse_max']} kt")
    print(f"   → Rafales: {result['vent_rafales']} kt")
    print(f"   → Direction: {result['vent_direction']} ({result['vent_direction_deg']}°)")

 TEST DU PARSER DE VENT

 Entrée: "10/15 kt atteignant 20/25 kt au nord d'Antalaha"
   → Min: 10 kt
   → Max: 15 kt
   → Rafales: 25 kt
   → Direction: E (90°)

 Entrée: "05/10 kt localement 15 kt"
   → Min: 5 kt
   → Max: 10 kt
   → Rafales: 15 kt
   → Direction: E (90°)

 Entrée: "15/20 kt localement 20 kt au sud de Majunga, variable 05/10k..."
   → Min: 15 kt
   → Max: 20 kt
   → Rafales: 20 kt
   → Direction: E (90°)

 Entrée: "secteur sud 20/25kt localement 30kt"
   → Min: 20 kt
   → Max: 25 kt
   → Rafales: 30 kt
   → Direction: SUD (180°)

 Entrée: "vent de Sud-est 15/20 kt"
   → Min: 15 kt
   → Max: 20 kt
   → Rafales: None kt
   → Direction: E (90°)

 Entrée: "Non spécifié"
   → Min: None kt
   → Max: None kt
   → Rafales: None kt
   → Direction: None (None°)


##  5. Standardisation de l'État de la Mer

### Échelle utilisée:
Conversion vers l'**échelle de Douglas** (1-9) basée sur les termes utilisés dans les bulletins.

In [8]:
class EtatMerParser:
    """
    Parse les descriptions d'état de la mer et les convertit
    en scores numériques (échelle de Douglas modifiée).
    """
    
    # Mapping des termes vers des scores
    # Score de 1 (calme) à 9 (énorme)
    ECHELLE = {
        # Score 1-2: Calme
        'calme': 1,
        'très calme': 1,
        
        # Score 3: Belle
        'belle': 3,
        
        # Score 4: Peu agitée
        'peu agitée': 4,
        'peu agitee': 4,
        'légèrement agitée': 4,
        
        # Score 5: Agitée
        'agitée': 5,
        'agitee': 5,
        
        # Score 6: Forte
        'forte': 6,
        
        # Score 7: Très forte
        'très forte': 7,
        'tres forte': 7,
        
        # Score 8: Grosse
        'grosse': 8,
        
        # Score 9: Énorme
        'énorme': 9,
        'enorme': 9
    }
    
    def parser(self, texte: str) -> Dict:
        """
        Parse une description d'état de la mer.
        
        Args:
            texte: Description textuelle
            
        Returns:
            Dict avec score_min, score_max, hauteur_vague estimée
        """
        resultat = {
            'mer_score_min': None,
            'mer_score_max': None,
            'mer_hauteur_min': None,
            'mer_hauteur_max': None,
            'mer_description': None
        }
        
        if not texte or pd.isna(texte) or texte == 'Non spécifié':
            return resultat
        
        texte_lower = str(texte).lower().strip()
        resultat['mer_description'] = texte_lower
        
        scores = []
        
        # Chercher les termes dans le texte
        for terme, score in self.ECHELLE.items():
            if terme in texte_lower:
                scores.append(score)
        
        if scores:
            resultat['mer_score_min'] = min(scores)
            resultat['mer_score_max'] = max(scores)
            
            # Estimer la hauteur de vague
            resultat['mer_hauteur_min'] = self._score_to_hauteur(min(scores))[0]
            resultat['mer_hauteur_max'] = self._score_to_hauteur(max(scores))[1]
        
        # Chercher des hauteurs explicites (ex: "Hauteur de vague 1.5m")
        match_hauteur = re.search(r'hauteur.*?(\d+(?:\.\d+)?)\s*(?:m|mètre)', texte_lower)
        if match_hauteur:
            hauteur = float(match_hauteur.group(1))
            resultat['mer_hauteur_max'] = hauteur
        
        return resultat
    
    def _score_to_hauteur(self, score: int) -> Tuple[float, float]:
        """Convertit un score en plage de hauteur de vague."""
        hauteurs = {
            1: (0, 0.1),
            2: (0, 0.1),
            3: (0.1, 0.5),
            4: (0.5, 1.25),
            5: (1.25, 2.5),
            6: (2.5, 4.0),
            7: (4.0, 6.0),
            8: (6.0, 9.0),
            9: (9.0, 14.0)
        }
        return hauteurs.get(score, (0, 0))

print(" Classe EtatMerParser chargée")

 Classe EtatMerParser chargée


In [9]:
parser_mer = EtatMerParser()

tests_mer = [
    "agitée à forte",
    "peu agitée à agitée",
    "belle à peu agitée",
    "forte, très forte près Morombe",
    "agitée à forte Hauteur de vague 1.5m",
    "Non spécifié"
]

print(" TEST DU PARSER D'ÉTAT DE LA MER")
print("=" * 70)

for test in tests_mer:
    result = parser_mer.parser(test)
    print(f"\n Entrée: \"{test}\"")
    print(f"   → Score: {result['mer_score_min']} - {result['mer_score_max']}")
    print(f"   → Hauteur: {result['mer_hauteur_min']} - {result['mer_hauteur_max']} m")

 TEST DU PARSER D'ÉTAT DE LA MER

 Entrée: "agitée à forte"
   → Score: 5 - 6
   → Hauteur: 1.25 - 4.0 m

 Entrée: "peu agitée à agitée"
   → Score: 4 - 5
   → Hauteur: 0.5 - 2.5 m

 Entrée: "belle à peu agitée"
   → Score: 3 - 5
   → Hauteur: 0.1 - 2.5 m

 Entrée: "forte, très forte près Morombe"
   → Score: 6 - 7
   → Hauteur: 2.5 - 6.0 m

 Entrée: "agitée à forte Hauteur de vague 1.5m"
   → Score: 5 - 6
   → Hauteur: 1.25 - 1.5 m

 Entrée: "Non spécifié"
   → Score: None - None
   → Hauteur: None - None m


##  6. Standardisation du Temps

### Variables extraites:
- **Présence de précipitations** (pluie, averses, orages)
- **Visibilité réduite** (brouillard, brume)
- **Score de conditions dangereuses**

In [10]:
==================
# CLASSE DE PARSING DU TEMPS
==================

class TempsParser:
    """
    Parse les descriptions du temps météorologique et extrait
    des indicateurs binaires et de sévérité.
    """
    
    # Mots-clés pour la détection
    PRECIPITATION = [
        'pluie', 'pluies', 'averses', 'averse',
        'précipitation', 'précipitations'
    ]
    
    ORAGES = [
        'orage', 'orages', 'orageux', 'tonnerre',
        'éclair', 'foudre', 'tempête'
    ]
    
    VISIBILITE_REDUITE = [
        'brouillard', 'brume', 'brumeux',
        'visibilité réduite', 'mauvaise visibilité'
    ]
    
    TEMPS_CLAIR = [
        'sec', 'beau', 'clair', 'ensoleillé',
        'dégagé', 'peu nuageux'
    ]
    
    NUAGEUX = [
        'nuageux', 'couvert', 'nuages'
    ]
    
    INTENSITE = {
        'faible': 1,
        'faibles': 1,
        'légère': 1,
        'légères': 1,
        'modéré': 2,
        'modérée': 2,
        'modérés': 2,
        'modérées': 2,
        'forte': 3,
        'fortes': 3,
        'violent': 4,
        'violente': 4,
        'violents': 4,
        'très forte': 4,
        'intense': 4
    }
    
    def parser(self, texte: str) -> Dict:
        """
        Parse une description du temps.
        
        Args:
            texte: Description textuelle
            
        Returns:
            Dict avec indicateurs du temps
        """
        resultat = {
            'temps_precipitation': 0,
            'temps_orage': 0,
            'temps_visibilite_reduite': 0,
            'temps_clair': 0,
            'temps_nuageux': 0,
            'temps_intensite': 0,
            'temps_score_danger': 0
        }
        
        if not texte or pd.isna(texte) or texte == 'Non spécifié':
            return resultat
        
        texte_lower = str(texte).lower().strip()
        
        # Détecter les précipitations
        for mot in self.PRECIPITATION:
            if mot in texte_lower:
                resultat['temps_precipitation'] = 1
                break
        
        # Détecter les orages
        for mot in self.ORAGES:
            if mot in texte_lower:
                resultat['temps_orage'] = 1
                break
        
        # Détecter la visibilité réduite
        for mot in self.VISIBILITE_REDUITE:
            if mot in texte_lower:
                resultat['temps_visibilite_reduite'] = 1
                break
        
        # Détecter le temps clair
        for mot in self.TEMPS_CLAIR:
            if mot in texte_lower:
                resultat['temps_clair'] = 1
                break
        
        # Détecter le temps nuageux
        for mot in self.NUAGEUX:
            if mot in texte_lower:
                resultat['temps_nuageux'] = 1
                break
        
        # Détecter l'intensité
        for mot, intensite in self.INTENSITE.items():
            if mot in texte_lower:
                resultat['temps_intensite'] = max(resultat['temps_intensite'], intensite)
        
        # Calculer le score de danger
        # Plus le score est élevé, plus les conditions sont dangereuses
        score = 0
        score += resultat['temps_precipitation'] * 1
        score += resultat['temps_orage'] * 3
        score += resultat['temps_visibilite_reduite'] * 2
        score += resultat['temps_intensite']
        resultat['temps_score_danger'] = score
        
        return resultat

print(" Classe TempsParser chargée")

 Classe TempsParser chargée


In [11]:
parser_temps = TempsParser()

tests_temps = [
    "Pluies faibles à modérées",
    "Temps sec",
    "Temps peu nuageux",
    "Orages violents avec averses",
    "Brouillard le matin, pluies faibles",
    "Temps partiellement nuageux",
    "Non spécifié"
]

print(" TEST DU PARSER DE TEMPS")
print("=" * 70)

for test in tests_temps:
    result = parser_temps.parser(test)
    print(f"\n Entrée: \"{test}\"")
    print(f"   → Précipitation: {result['temps_precipitation']}")
    print(f"   → Orage: {result['temps_orage']}")
    print(f"   → Visibilité réduite: {result['temps_visibilite_reduite']}")
    print(f"   → Intensité: {result['temps_intensite']}")
    print(f"   → Score danger: {result['temps_score_danger']}")

 TEST DU PARSER DE TEMPS

 Entrée: "Pluies faibles à modérées"
   → Précipitation: 1
   → Orage: 0
   → Visibilité réduite: 0
   → Intensité: 2
   → Score danger: 3

 Entrée: "Temps sec"
   → Précipitation: 0
   → Orage: 0
   → Visibilité réduite: 0
   → Intensité: 0
   → Score danger: 0

 Entrée: "Temps peu nuageux"
   → Précipitation: 0
   → Orage: 0
   → Visibilité réduite: 0
   → Intensité: 0
   → Score danger: 0

 Entrée: "Orages violents avec averses"
   → Précipitation: 1
   → Orage: 1
   → Visibilité réduite: 0
   → Intensité: 4
   → Score danger: 8

 Entrée: "Brouillard le matin, pluies faibles"
   → Précipitation: 1
   → Orage: 0
   → Visibilité réduite: 1
   → Intensité: 1
   → Score danger: 4

 Entrée: "Temps partiellement nuageux"
   → Précipitation: 0
   → Orage: 0
   → Visibilité réduite: 0
   → Intensité: 0
   → Score danger: 0

 Entrée: "Non spécifié"
   → Précipitation: 0
   → Orage: 0
   → Visibilité réduite: 0
   → Intensité: 0
   → Score danger: 0


##  7. Application aux Données Complètes

Appliquons maintenant tous les parsers à l'ensemble des données météo.

In [12]:
print(" STANDARDISATION DES DONNÉES MÉTÉO")
print("=" * 70)
print(f"Traitement de {len(df_meteo)} enregistrements...\n")

# Initialiser les parsers
parser_vent = VentParser()
parser_mer = EtatMerParser()
parser_temps = TempsParser()

# Créer les nouvelles colonnes
colonnes_vent = ['vent_vitesse_min', 'vent_vitesse_max', 'vent_rafales', 'vent_direction', 'vent_direction_deg']
colonnes_mer = ['mer_score_min', 'mer_score_max', 'mer_hauteur_min', 'mer_hauteur_max']
colonnes_temps = ['temps_precipitation', 'temps_orage', 'temps_visibilite_reduite', 
                  'temps_clair', 'temps_nuageux', 'temps_intensite', 'temps_score_danger']

# Initialiser les colonnes
for col in colonnes_vent + colonnes_mer + colonnes_temps:
    df_meteo[col] = None

# Appliquer les parsers ligne par ligne
for idx, row in df_meteo.iterrows():
    # Parser le vent
    vent_data = parser_vent.parser(row['vent'])
    for col, val in vent_data.items():
        df_meteo.at[idx, col] = val
    
    # Parser l'état de la mer
    mer_data = parser_mer.parser(row['etat_mer'])
    for col in colonnes_mer:
        df_meteo.at[idx, col] = mer_data.get(col)
    
    # Parser le temps
    temps_data = parser_temps.parser(row['temps'])
    for col, val in temps_data.items():
        df_meteo.at[idx, col] = val

print(" Standardisation terminée!")
print(f"\n Nouvelles colonnes ajoutées: {len(colonnes_vent + colonnes_mer + colonnes_temps)}")

 STANDARDISATION DES DONNÉES MÉTÉO
Traitement de 1346 enregistrements...

 Standardisation terminée!

 Nouvelles colonnes ajoutées: 16


In [13]:
print(" APERÇU DES DONNÉES STANDARDISÉES")
print("=" * 70)

# Colonnes originales vs nouvelles
print("\n Colonnes du DataFrame:")
print(list(df_meteo.columns))

# Aperçu
print("\n Échantillon (colonnes vent):")
display(df_meteo[['date', 'zone', 'vent', 'vent_vitesse_min', 'vent_vitesse_max', 'vent_rafales']].head(10))

print("\n Échantillon (colonnes mer):")
display(df_meteo[['date', 'zone', 'etat_mer', 'mer_score_min', 'mer_score_max', 'mer_hauteur_max']].head(10))

print("\n Échantillon (colonnes temps):")
display(df_meteo[['date', 'zone', 'temps', 'temps_precipitation', 'temps_orage', 'temps_score_danger']].head(10))

 APERÇU DES DONNÉES STANDARDISÉES

 Colonnes du DataFrame:
['date', 'zone', 'vent', 'etat_mer', 'temps', 'vent_vitesse_min', 'vent_vitesse_max', 'vent_rafales', 'vent_direction', 'vent_direction_deg', 'mer_score_min', 'mer_score_max', 'mer_hauteur_min', 'mer_hauteur_max', 'temps_precipitation', 'temps_orage', 'temps_visibilite_reduite', 'temps_clair', 'temps_nuageux', 'temps_intensite', 'temps_score_danger']

 Échantillon (colonnes vent):


,date,zone,vent,vent_vitesse_min,vent_vitesse_max,vent_rafales
0,06/09/2019,CAP D'AMBRE A MAHANORO,10/15 kt atteignant 20/25 kt au nord d'Antalaha,10,15,25
1,06/09/2019,MAHANORO AU CAP SAINTE MARIE,05/10 kt devenant progressivement secteur sud ...,5,10,30
2,06/09/2019,CAP D'AMBRE A BESALAMPY,"15/20 kt localement 20 kt au sud de Majunga, v...",15,20,20
3,06/09/2019,BESALAMPY A MOROMBE,15/20 kt,15,20,None
4,06/09/2019,MOROMBE A CAP SAINTE MARIE,20/25 kt atteignant 30/35 kt entre Morombe et,20,25,35
5,07/09/2019,CAP D'AMBRE A BESALAMPY,DE SUD-EST 15/20 KT ATTEIGNANT 30 KT LE MATIN ...,15,20,30
6,07/09/2019,BESALAMPY A MOROMBE,DE SECTEUR SUD 20/25 KT. LOCALEMENT 30 KT AU C...,20,25,30
7,07/09/2019,MOROMBE AU CAP SAINTE MARIE,DE SUD-EST 20/25 KT TOURNANT SECTEUR EST TEMPO...,20,25,None
8,07/09/2019,MAHANORO AU CAP SAINTE MARIE,DE SUD-EST 20/25 KT ATTEIGNANT LOCALEMENT 30KT,20,25,30
9,07/09/2019,CAP D'AMBRE A MAHANORO,DE SUD-EST 15/20 KT ATTEIGNANT 25/30 KT AU NOR...,15,20,30



 Échantillon (colonnes mer):


,date,zone,etat_mer,mer_score_min,mer_score_max,mer_hauteur_max
0,06/09/2019,CAP D'AMBRE A MAHANORO,agitée à forte,5,6,4.0
1,06/09/2019,MAHANORO AU CAP SAINTE MARIE,agitée à forte,5,6,4.0
2,06/09/2019,CAP D'AMBRE A BESALAMPY,Non spécifié,None,None,None
3,06/09/2019,BESALAMPY A MOROMBE,"agitée a forte, très forte près Morombe dans l...",5,7,6.0
4,06/09/2019,MOROMBE A CAP SAINTE MARIE,Non spécifié,None,None,None
5,07/09/2019,CAP D'AMBRE A BESALAMPY,PEU AGITÉ,None,None,None
6,07/09/2019,BESALAMPY A MOROMBE,AGITÉ,None,None,None
7,07/09/2019,MOROMBE AU CAP SAINTE MARIE,F,None,None,None
8,07/09/2019,MAHANORO AU CAP SAINTE MARIE,"AGITÉE À FORTE, TRES FORTE PAR",5,7,6.0
9,07/09/2019,CAP D'AMBRE A MAHANORO,AGITÉE À FORTE. HAUTEUR DE VAGUE 2.8/3.2M,5,6,3.2



 Échantillon (colonnes temps):


,date,zone,temps,temps_precipitation,temps_orage,temps_score_danger
0,06/09/2019,CAP D'AMBRE A MAHANORO,Pluies faible a modérée,1,0,3
1,06/09/2019,MAHANORO AU CAP SAINTE MARIE,pluies,1,0,1
2,06/09/2019,CAP D'AMBRE A BESALAMPY,Temps sec,0,0,0
3,06/09/2019,BESALAMPY A MOROMBE,Temps sec,0,0,0
4,06/09/2019,MOROMBE A CAP SAINTE MARIE,Temps partiellement nuageux,0,0,0
5,07/09/2019,CAP D'AMBRE A BESALAMPY,Non specifie,0,0,0
6,07/09/2019,BESALAMPY A MOROMBE,SEC,0,0,0
7,07/09/2019,MOROMBE AU CAP SAINTE MARIE,PARTIELLEMENT NUAGEUX,0,0,0
8,07/09/2019,MAHANORO AU CAP SAINTE MARIE,Non specifie,0,0,0
9,07/09/2019,CAP D'AMBRE A MAHANORO,Non specifie,0,0,0


##  8. Statistiques des Variables Standardisées

In [14]:
print(" STATISTIQUES DES VARIABLES STANDARDISÉES")
print("=" * 70)

# Variables numériques
vars_numeriques = ['vent_vitesse_min', 'vent_vitesse_max', 'vent_rafales',
                   'mer_score_min', 'mer_score_max', 'mer_hauteur_max',
                   'temps_score_danger']

print("\n Statistiques descriptives:")
display(df_meteo[vars_numeriques].describe())

 STATISTIQUES DES VARIABLES STANDARDISÉES

 Statistiques descriptives:


,vent_vitesse_min,vent_vitesse_max,vent_rafales,mer_score_min,mer_score_max,mer_hauteur_max,temps_score_danger
count,704,704,718,1093,1093,1093.0,1346
unique,5,6,10,4,5,29.0,8
top,5,10,25,5,5,2.5,0
freq,327,289,228,497,715,608.0,835


In [15]:
print("\n TAUX DE REMPLISSAGE DES VARIABLES")
print("=" * 70)

toutes_nouvelles_colonnes = colonnes_vent + colonnes_mer + colonnes_temps

remplissage = []
for col in toutes_nouvelles_colonnes:
    non_null = df_meteo[col].notna().sum()
    pct = 100 * non_null / len(df_meteo)
    remplissage.append({
        'Variable': col,
        'Valeurs remplies': non_null,
        'Pourcentage': f"{pct:.1f}%"
    })

df_remplissage = pd.DataFrame(remplissage)
display(df_remplissage)


 TAUX DE REMPLISSAGE DES VARIABLES


,Variable,Valeurs remplies,Pourcentage
0,vent_vitesse_min,704,52.3%
1,vent_vitesse_max,704,52.3%
2,vent_rafales,718,53.3%
3,vent_direction,1159,86.1%
4,vent_direction_deg,1159,86.1%
5,mer_score_min,1093,81.2%
6,mer_score_max,1093,81.2%
7,mer_hauteur_min,1093,81.2%
8,mer_hauteur_max,1093,81.2%
9,temps_precipitation,1346,100.0%


##  9. Création du Score de Risque Composite

Créons un **score de risque global** combinant toutes les variables météo.

In [16]:
==================
# CALCUL DU SCORE DE RISQUE COMPOSITE
==================

def calculer_score_risque(row) -> float:
    """
    Calcule un score de risque composite basé sur les conditions météo.
    
    Score de 0 (conditions idéales) à 100 (conditions très dangereuses)
    """
    score = 0
    
    # === CONTRIBUTION DU VENT (max 40 points) ===
    # Vitesse du vent (max 25 points)
    vitesse_max = row.get('vent_vitesse_max')
    if pd.notna(vitesse_max):
        if vitesse_max < 10:
            score += 0
        elif vitesse_max < 15:
            score += 5
        elif vitesse_max < 20:
            score += 10
        elif vitesse_max < 25:
            score += 15
        elif vitesse_max < 30:
            score += 20
        else:
            score += 25
    
    # Rafales (max 15 points)
    rafales = row.get('vent_rafales')
    if pd.notna(rafales):
        if rafales >= 35:
            score += 15
        elif rafales >= 30:
            score += 10
        elif rafales >= 25:
            score += 5
    
    # === CONTRIBUTION DE LA MER (max 40 points) ===
    score_mer = row.get('mer_score_max')
    if pd.notna(score_mer):
        # Échelle 1-9, convertie en 0-40
        score += (score_mer - 1) * 5
    
    # === CONTRIBUTION DU TEMPS (max 20 points) ===
    score_temps = row.get('temps_score_danger', 0)
    if pd.notna(score_temps):
        # Score danger max ~7, convertir en 0-20
        score += min(score_temps * 3, 20)
    
    return min(score, 100)

# Appliquer le calcul
df_meteo['score_risque'] = df_meteo.apply(calculer_score_risque, axis=1)

print(" SCORE DE RISQUE CALCULÉ")
print("=" * 70)
print("\n Distribution du score de risque:")
print(df_meteo['score_risque'].describe())

print("\n Répartition par catégories:")
bins = [0, 20, 40, 60, 80, 100]
labels = ['Faible (0-20)', 'Modéré (20-40)', 'Élevé (40-60)', 'Très élevé (60-80)', 'Extrême (80-100)']
df_meteo['categorie_risque'] = pd.cut(df_meteo['score_risque'], bins=bins, labels=labels)
print(df_meteo['categorie_risque'].value_counts())

 SCORE DE RISQUE CALCULÉ

 Distribution du score de risque:
count    1346.000000
mean       26.569837
std        12.218577
min         0.000000
25%        20.000000
50%        25.000000
75%        35.000000
max        72.000000
Name: score_risque, dtype: float64

 Répartition par catégories:
categorie_risque
Modéré (20-40)        831
Faible (0-20)         330
Élevé (40-60)         130
Très élevé (60-80)      5
Extrême (80-100)        0
Name: count, dtype: int64


##  10. Sauvegarde des Données Standardisées

In [17]:
# Fichier de sortie
output_file = os.path.join(FINAL_PATH, 'meteo_standardise.csv')

# Sauvegarder
df_meteo.to_csv(output_file, index=False, encoding='utf-8')

print(" SAUVEGARDE EFFECTUÉE")
print("=" * 70)
print(f"Fichier: {output_file}")
print(f"Nombre d'enregistrements: {len(df_meteo)}")
print(f"Nombre de colonnes: {len(df_meteo.columns)}")
print(f"Taille: {os.path.getsize(output_file) / 1024:.1f} KB")

print("\n Colonnes du fichier final:")
for i, col in enumerate(df_meteo.columns, 1):
    print(f"   {i:2d}. {col}")

 SAUVEGARDE EFFECTUÉE
Fichier: /home/henintsoa/CFIM/final/data/meteo_standardise.csv
Nombre d'enregistrements: 1346
Nombre de colonnes: 23
Taille: 255.9 KB

 Colonnes du fichier final:
    1. date
    2. zone
    3. vent
    4. etat_mer
    5. temps
    6. vent_vitesse_min
    7. vent_vitesse_max
    8. vent_rafales
    9. vent_direction
   10. vent_direction_deg
   11. mer_score_min
   12. mer_score_max
   13. mer_hauteur_min
   14. mer_hauteur_max
   15. temps_precipitation
   16. temps_orage
   17. temps_visibilite_reduite
   18. temps_clair
   19. temps_nuageux
   20. temps_intensite
   21. temps_score_danger
   22. score_risque
   23. categorie_risque



                   PHASE 1 TERMINÉE AVEC SUCCÈS !                   
                                                                     
  FICHIERS PRODUITS:                                               
                                                                     
 1. meteo_marine_cotiere_complet.csv                                
    └─ Données météo consolidées 2019-2022                          
                                                                     
 2. correspondance_zones_regions.csv                                
    └─ Mapping zones côtières ↔ régions administratives             
                                                                     
 3. mapping_zones_regions.json                                      
    └─ Mapping complet en format JSON                               
                                                                     
 4. meteo_standardise.csv                                           
    └─ Données météo avec variables numériques                      
                                                                
                                                                     
  VARIABLES CRÉÉES:                                                
                                                                     
 • Vent: 5 variables (vitesse min/max, rafales, direction)          
 • Mer: 4 variables (score Douglas, hauteur vague)                  
 • Temps: 7 variables (précipitation, orage, visibilité...)         
 • Risque: 2 variables (score composite, catégorie)                 
                                                                
                                                                     
  PROCHAINE PHASE:                                                 
                                                                     
 Phase 2: Enrichissement des données                                
   • Fusion incidents + météo                                       
   • Création du dataset d'entraînement                             
   • Intégration données satellites (optionnel)                     
                                            